# Chemistry PDF extractor (Kaggle)

**Inputs**
- **Code**: cloned from your public GitHub repo (Internet required).
- **PDFs**: one Kaggle dataset (zip of `.pdf` files at the top level).

**Notebook settings**: GPU + Internet.

1. Push this project to a **public** GitHub repository.
2. Create a Kaggle **dataset** that contains only your PDF files.
3. Edit `GITHUB_REPO` and `PDF_INPUT_DIR` in the next cell, then run all cells.

In [ ]:
!apt-get update -qq && apt-get install -y -qq poppler-utils git

In [ ]:
import os
import sys
from pathlib import Path

# --- Edit these two lines ---
GITHUB_REPO = "https://github.com/YOUR_USERNAME/test-paper-extractor.git"
GITHUB_BRANCH = "main"
PDF_INPUT_DIR = Path("/kaggle/input/chemistry-papers")  # slug from Add Data

PROJECT_ROOT = Path("/kaggle/working/test-paper-extractor")
OUT_ROOT = Path("/kaggle/working/out")
DPI = 300
QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

print("Datasets mounted:", [p.name for p in Path("/kaggle/input").iterdir()])

In [ ]:
import shutil
import subprocess

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

clone_cmd = [
    "git", "clone", "--depth", "1",
    "--branch", GITHUB_BRANCH,
    GITHUB_REPO,
    str(PROJECT_ROOT),
]
subprocess.check_call(clone_cmd)

if not (PROJECT_ROOT / "extract_paper.py").is_file():
    raise FileNotFoundError(f"Repo cloned but extract_paper.py missing under {PROJECT_ROOT}")

pdfs = sorted(PDF_INPUT_DIR.glob("*.pdf"))
if not pdfs:
    raise FileNotFoundError(
        f"No PDFs in {PDF_INPUT_DIR}. Put .pdf files at the dataset root and fix PDF_INPUT_DIR."
    )
print(f"Cloned {GITHUB_REPO}")
print(f"Found {len(pdfs)} PDF(s) in {PDF_INPUT_DIR}")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

!pip install -q -r requirements.txt

In [ ]:
from batch_extract import main as batch_main

OUT_ROOT.mkdir(parents=True, exist_ok=True)
argv = [
    str(PDF_INPUT_DIR),
    "--out-root", str(OUT_ROOT),
    "--",
    "--backend", "qwen_hf",
    "--dpi", str(DPI),
    "--qwen-model", QWEN_MODEL,
    "--save-page-images",
]
batch_main(argv)

In [ ]:
from preview import build_preview_html

for questions_path in OUT_ROOT.glob("*/questions.json"):
    print("Preview:", build_preview_html(questions_path))

In [ ]:
import shutil

zip_path = Path("/kaggle/working/out.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", OUT_ROOT)
print("Download from Output:", zip_path)